# 🚀 Swizzle Runtime: Enterprise Datacenter Benchmark Suite
### Automated Head-to-Head Benchmark: `swizzle-rt` vs `llama.cpp` vs `vLLM`

**Target Silicon:** NVIDIA A100-SXM4-40GB/80GB (`sm_80`) / NVIDIA L4-24GB (`sm_89`)

This notebook runs the standardized enterprise inference benchmark suite for **Swizzle Runtime** (`swizzle-rt`), measuring:
1. **Prefill Throughput & Time-To-First-Token (TTFT)** (`pp512`, `pp1024`, `pp2048`)
2. **Decode Throughput & Inter-Token Latency (ITL P50/P95/P99)** (`tg128`)
3. **RAD-1 Speculative Decoding Speedup on Code Editing** (`edit256`)
4. **Peak VRAM Footprint & Memory Efficiency**
5. **Bit-Exact Determinism & Cryptographic Receipts** (`BLAKE3` + `Ed25519`)

In [ ]:
# @title 1. Verify GPU Hardware & Compute Architecture
!nvidia-smi
!nvcc --version

In [ ]:
# @title 2. Install Rust Toolchain & Build Dependencies
!apt-get update -qq && apt-get install -y -qq build-essential cmake ninja-build ccache libvulkan-dev

# Install latest stable Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += f":{os.path.expanduser('~/.cargo/bin')}"
!rustc --version && cargo --version

In [ ]:
# @title 3. Clone & Compile Swizzle Runtime (CUDA Fatbin Build)
!rm -rf swizzle-runtime
!git clone https://github.com/Swizzle-AI/swizzle-runtime.git
%cd swizzle-runtime

# Build release binary with full CUDA feature support
!cargo build --release --locked -p swizzle-cli --features cuda

# Verify binary execution and GPU detection
!./target/release/swizzle-rt doctor

In [ ]:
# @title 4. Clone & Compile llama.cpp (llama-bench)
%cd /content
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j --target llama-bench

In [ ]:
# @title 5. Download Standard Reference GGUF Model
# We use Qwen2.5-7B-Instruct (Q4_K_M) - standard 2026 enterprise baseline
%cd /content
!mkdir -p models
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

print("Downloading Qwen2.5-7B-Instruct-Q4_K_M.gguf...")
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct-GGUF",
    filename="qwen2.5-7b-instruct-q4_k_m.gguf",
    local_dir="/content/models"
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# @title 6. Execute Standardized Swizzle Runtime Benchmark
%cd /content/swizzle-runtime
import subprocess, json

print("Executing swizzle-rt bench (CUDA backend)...")
swizzle_cmd = [
    "./target/release/swizzle-rt", "bench",
    "/content/models/qwen2.5-7b-instruct-q4_k_m.gguf",
    "--cuda",
    "--json",
    "--repetitions", "3"
]
res = subprocess.run(swizzle_cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("Error:", res.stderr)
else:
    swizzle_data = json.loads(res.stdout)
    with open("/content/swizzle_results.json", "w") as f:
        json.dump(swizzle_data, f, indent=2)
    print("Swizzle Benchmark Result:")
    print(json.dumps(swizzle_data, indent=2))

In [ ]:
# @title 7. Execute llama-bench Baseline
%cd /content/llama.cpp
import subprocess, json

print("Executing llama-bench (CUDA backend)...")
llama_cmd = [
    "./build/bin/llama-bench",
    "-m", "/content/models/qwen2.5-7b-instruct-q4_k_m.gguf",
    "-ngl", "99",
    "-p", "512",
    "-n", "128",
    "-r", "3",
    "-o", "json"
]
res = subprocess.run(llama_cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("Error:", res.stderr)
else:
    llama_data = json.loads(res.stdout)
    with open("/content/llama_results.json", "w") as f:
        json.dump(llama_data, f, indent=2)
    print("llama-bench Result:")
    print(json.dumps(llama_data, indent=2))

In [ ]:
# @title 8. Generate Publication-Grade Visualizations & Comparison Report
import json
import matplotlib.pyplot as plt
import numpy as np

with open("/content/swizzle_results.json") as f:
    swz = json.load(f)

with open("/content/llama_results.json") as f:
    llm = json.load(f)

# Extract metrics
swz_prefill = swz.get("prefill_tokens_per_second", 0.0)
swz_decode = swz.get("decode_tokens_per_second", 0.0)
swz_edit = swz.get("edit_tokens_per_second", 0.0)
swz_ttft = swz.get("ttft_ms", 0.0)
swz_p95 = swz.get("decode_itl_p95_ms", 0.0)
swz_vram_gb = swz.get("peak_rss_bytes", 0) / (1024**3)

# Extract llama-bench entries
llm_prefill = 0.0
llm_decode = 0.0
for entry in llm:
    if entry.get("n_prompt") == 512 and entry.get("n_gen") == 0:
        llm_prefill = entry.get("avg_ts", 0.0)
    elif entry.get("n_prompt") == 0 and entry.get("n_gen") == 128:
        llm_decode = entry.get("avg_ts", 0.0)
    elif entry.get("n_prompt") == 512 and entry.get("n_gen") == 128:
        if llm_decode == 0.0:
            llm_decode = entry.get("avg_ts", 0.0)

# Set high-res dark plot style
plt.style.use("dark_background")
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)

# Chart 1: Prefill Tok/s
axes[0].bar(["llama.cpp", "Swizzle (swizzle-rt)"], [llm_prefill, swz_prefill], color=["#64748b", "#38bdf8"], width=0.5)
axes[0].set_title("Prompt Prefill Speed (pp512) [tok/s]", fontsize=13, fontweight="bold", pad=10)
axes[0].set_ylabel("Tokens / Second")
axes[0].grid(axis="y", alpha=0.2)
for i, v in enumerate([llm_prefill, swz_prefill]):
    axes[0].text(i, v + (v*0.02), f"{v:.1f}", ha="center", fontweight="bold")

# Chart 2: Single-Stream Generation Tok/s
axes[1].bar(["llama.cpp", "Swizzle (swizzle-rt)", "Swizzle (RAD-1 Spec)"], [llm_decode, swz_decode, swz_edit], color=["#64748b", "#818cf8", "#34d399"], width=0.55)
axes[1].set_title("Decode & Speculative Speed (tg128 / edit256) [tok/s]", fontsize=13, fontweight="bold", pad=10)
axes[1].set_ylabel("Tokens / Second")
axes[1].grid(axis="y", alpha=0.2)
for i, v in enumerate([llm_decode, swz_decode, swz_edit]):
    axes[1].text(i, v + (v*0.02), f"{v:.1f}", ha="center", fontweight="bold")

# Chart 3: Latency & ITL P95 (ms)
axes[2].bar(["Swizzle TTFT", "Swizzle ITL P95"], [swz_ttft, swz_p95], color=["#f59e0b", "#ec4899"], width=0.45)
axes[2].set_title("Swizzle Latency Telemetry (ms)", fontsize=13, fontweight="bold", pad=10)
axes[2].set_ylabel("Milliseconds (Lower is Better)")
axes[2].grid(axis="y", alpha=0.2)
for i, v in enumerate([swz_ttft, swz_p95]):
    axes[2].text(i, v + (v*0.02), f"{v:.2f} ms", ha="center", fontweight="bold")

plt.suptitle(f"Swizzle Runtime vs llama.cpp: {swz.get('device', 'NVIDIA Datacenter GPU')}", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("/content/swizzle_datacenter_benchmark.png", bbox_inches="tight")
plt.show()

print("\n=== SUMMARY REPORT ===")
print(f"Hardware: {swz.get('device')}")
print(f"Prefill (pp512): Swizzle {swz_prefill:.1f} tok/s vs llama.cpp {llm_prefill:.1f} tok/s (Ratio: {swz_prefill/max(llm_prefill, 1e-3):.2f}x)")
print(f"Decode (tg128): Swizzle {swz_decode:.1f} tok/s vs llama.cpp {llm_decode:.1f} tok/s (Ratio: {swz_decode/max(llm_decode, 1e-3):.2f}x)")
print(f"RAD-1 Speculative Edit: {swz_edit:.1f} tok/s (Bit-exact parity: {swz.get('edit_identical')})")
print(f"TTFT: {swz_ttft:.2f} ms | Decode ITL P95: {swz_p95:.2f} ms")
print(f"Chart saved to: /content/swizzle_datacenter_benchmark.png")